In [ ]:
# === Setup (Part 2) — keep byte-identical across RQ notebooks ===
import json
import os
import pickle
from pathlib import Path

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

CONFIG_PATH = (Path("..") / "config.json").resolve()
with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)

PROJECT_ROOT = CONFIG_PATH.parent


def _expand_tree(obj):
    """Expand ~ in all string paths; leave non-strings / null unchanged."""
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _model_src(key: str) -> str:
    """Return local path or HF hub id for a model key in CFG['models']."""
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def load_generative(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    return tokenizer, model


def load_encoder(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModel.from_pretrained(src)
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = cui_pool.get("n_cuis", len(cui_pool.get("cuis", {})))
print(f"Config: {CONFIG_PATH}")
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
if _n_cuis < 10_000:
    print("WARNING: CUI count looks like the MeSH subset (~1,201), not full UMLS.")
else:
    print("Confirmed: full_umls-scale CUI pool loaded.")


# RQ3: Domain Adaptation and Semantic Stability
## SIT723 - masters Research Techniques and Applications

**Research Question:**
Does domain-specific biomedical pretraining reduce semantic entropy
relative to general-purpose models at equivalent parameter scale,
and does this stability advantage generalise from medical concept
normalisation to biomedical and general-domain question answering?

**Sub-questions:**
- **SQ1:** Do biomedical models show lower semantic entropy than
  general-purpose models at equivalent scale?
  - Pair 1: BioBERT vs BERT-base (both 110M parameters)
  - Pair 2: BioMistral-7B vs FLAN-T5-XXL (large-scale tier)
- **SQ2:** Does any stability advantage persist on BioASQ
  (biomedical QA) and attenuate on SQuAD 2.0 (general-domain QA)?
- **SQ3:** Does domain adaptation improve semantic consistency, or
  does it only improve benchmark accuracy?

**Statistical operationalisation:**
- Primary: One-tailed Mann-Whitney U tests, two within-scale pairs
  Pre-specified threshold: rank-biserial r ≥ 0.30 per pair
- Generalisation: Effect size attenuation on SQuAD 2.0 vs
  MedMentions
- BH-FDR correction at q = 0.05

**Datasets:**
- MedMentions ST21pv: N=550 (reused from RQ1/RQ2)
- BioASQ Task B: N=150 (new)
- SQuAD 2.0: N=200 (new)

**Gap addressed:** Gap 3 - Biomedical LLMs have not been
systematically compared for semantic stability under controlled
meaning-preserving perturbation. The within-scale design isolates
domain-specific pretraining from the confound of model scale.

## 1) Environment Setup

All dependencies, random seed, GPU detection, and configuration.
RQ3 reuses MedMentions entropy from RQ1/RQ2 outputs and adds
BioASQ and SQuAD 2.0 as new datasets.

In [ ]:
import os, sys, math, json, random, warnings, gc, time, re
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import mannwhitneyu, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import (
    AutoTokenizer, AutoModel,
    T5ForConditionalGeneration,
    AutoModelForCausalLM,
    MarianMTModel, MarianTokenizer,
    pipeline,
)
try:
    from datasets import load_dataset
except ModuleNotFoundError:
    print("[setup] Installing missing dependency: datasets")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets"])
    from datasets import load_dataset

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
              f"Free: {free/1e9:.1f}GB / {total/1e9:.1f}GB")

sns.set_theme(style="whitegrid", context="notebook")

# Config
N_BIOASQ       = 150
N_SQUAD        = 200
K_PERTURB      = 8
BOOTSTRAP_B    = 1000
USE_INT8       = True
DO_SAMPLE      = False
TEMPERATURE    = 1.0
MAX_NEW_TOKENS = 64

# Pre-registered within-scale pairs
PAIR_1 = ("BioBERT",       "BERT-base")
PAIR_2 = ("BioMistral-7B", "FLAN-T5-XXL")

# Paths
PROJECT_ROOT = Path("/home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs")
RQ1_INTER    = PROJECT_ROOT / "outputs" / "rq1" / "intermediate"
RQ2_TABLES   = PROJECT_ROOT / "outputs" / "rq2" / "tables"
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "rq3"
FIGURES_DIR  = OUTPUT_DIR / "figures"
TABLES_DIR   = OUTPUT_DIR / "tables"
INTER_DIR    = OUTPUT_DIR / "intermediate"
for p in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, INTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Model specs
ENCODER_SPECS = {
    "BERT-base": {
        "hf_id":   "bert-base-uncased",
        "domain":  "general",
        "params":  "110M",
    },
    "BioBERT": {
        "hf_id":   "dmis-lab/biobert-v1.1",
        "domain":  "biomedical",
        "params":  "110M",
    },
    "PubMedBERT": {
        "hf_id":   "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
        "domain":  "biomedical",
        "params":  "110M",
    },
}
GEN_SPECS = {
    "FLAN-T5-base": {
        "hf_id":  "google/flan-t5-base",
        "type":   "seq2seq",
        "int8":   False,
        "domain": "general",
        "params": "250M",
    },
    "FLAN-T5-XXL": {
        "hf_id":  "google/flan-t5-xxl",
        "type":   "seq2seq",
        "int8":   USE_INT8,
        "domain": "general",
        "params": "11B",
    },
    "BioMistral-7B": {
        "hf_id":  "BioMistral/BioMistral-7B",
        "type":   "causal",
        "int8":   USE_INT8,
        "domain": "biomedical",
        "params": "7B",
    },
}

print(f"\nPair 1: {PAIR_1[0]} vs {PAIR_1[1]} (110M)")
print(f"Pair 2: {PAIR_2[0]} vs {PAIR_2[1]} (large-scale)")
print(f"BioASQ N={N_BIOASQ} | SQuAD N={N_SQUAD} | "
      f"K_PERTURB={K_PERTURB} | Bootstrap B={BOOTSTRAP_B}")

## 2) Load MedMentions Entropy (Reused from RQ1/RQ2)

Loads pre-computed entropy scores from RQ2 for MedMentions ST21pv
N=550. This is the clinical anchor point of the domain-continuum
analysis. No re-computation needed.

In [ ]:
df_rq2 = pd.read_csv(RQ2_TABLES / "rq2_all_entropy.csv")
df_rq2["dataset"] = "MedMentions"

_model_meta = {
    **{k: {"domain": v["domain"], "params": v["params"]}
       for k, v in ENCODER_SPECS.items()},
    **{k: {"domain": v["domain"], "params": v["params"]}
       for k, v in GEN_SPECS.items()},
}
df_rq2["domain"] = df_rq2["model_name"].map(
    lambda x: _model_meta.get(x, {}).get("domain", "unknown"))
df_rq2["params"] = df_rq2["model_name"].map(
    lambda x: _model_meta.get(x, {}).get("params", "unknown"))

print(f"MedMentions rows loaded: {len(df_rq2)}")
print(f"Models: {df_rq2['model_name'].unique().tolist()}")
print("\nMedMentions entropy summary:")
print(df_rq2.groupby(["model_name","architecture"])[
    ["normalised_entropy","mean_accuracy"]
].agg(["mean","std"]).round(4).to_string())

## 3) Load BioASQ and SQuAD 2.0 Datasets

**BioASQ Task B (N=150):** Biomedical QA - tests whether a
stability advantage on medical concept normalisation persists on
biomedical question answering.

**SQuAD 2.0 (N=200):** General-domain QA - tests whether any
advantage attenuates when moving completely away from clinical text.

Both use the same k=8 perturbation pipeline and six-gate
validation as RQ1 to ensure comparability.

In [ ]:
# ── BioASQ - load from local zip ─────────────────────────────────────────
import zipfile, json as _json

_bioasq_zip = PROJECT_ROOT / "Datasets" / "BioASQ-training13b.zip"
_bioasq_df  = pd.DataFrame()

if _bioasq_zip.exists():
    print(f"Loading BioASQ from: {_bioasq_zip}")
    with zipfile.ZipFile(_bioasq_zip, "r") as zf:
        # Find the training JSON file inside the zip
        _json_files = [f for f in zf.namelist()
                       if f.endswith(".json") and "training" in f.lower()]
        print(f"JSON files found: {_json_files}")
        _json_file = _json_files[0] if _json_files else None

        if _json_file:
            with zf.open(_json_file) as fh:
                _bioasq_data = _json.load(fh)

    # BioASQ format: {"questions": [{"body": "...", "ideal_answer": [...]}]}
    _rows = []
    for q in _bioasq_data.get("questions", []):
        body   = q.get("body", "")
        answer = q.get("ideal_answer", [""])
        if isinstance(answer, list):
            answer = answer[0] if answer else ""
        if body and answer:
            _rows.append({
                "question":    body,
                "gold_answer": str(answer)[:200],
            })

    print(f"Total BioASQ questions: {len(_rows)}")

    _bioasq_df = pd.DataFrame(_rows).dropna().sample(
        n=min(N_BIOASQ, len(_rows)), random_state=SEED
    ).reset_index(drop=True)

    _bioasq_df["instance_id"]     = [f"bioasq_{i}"
                                      for i in range(len(_bioasq_df))]
    _bioasq_df["mention_context"] = _bioasq_df["question"]
    _bioasq_df["gold_mention"]    = _bioasq_df["gold_answer"].str[:80]
    _bioasq_df["gold_cui"]        = "QA_ANSWER"
    _bioasq_df.to_csv(INTER_DIR / "rq3_bioasq_instances.csv", index=False)
    print(f"BioASQ sampled: {len(_bioasq_df)} instances")
    print(_bioasq_df[["instance_id","question","gold_mention"]].head(3).to_string())
else:
    print(f"[ERROR] BioASQ zip not found at: {_bioasq_zip}")
    _bioasq_df = pd.DataFrame()

# ── SQuAD 2.0 ─────────────────────────────────────────────────────────────
_squad_raw = load_dataset("rajpurkar/squad_v2", split="validation")
_squad_ans = [r for r in _squad_raw if len(r["answers"]["text"]) > 0]
_rng_sq    = np.random.default_rng(SEED)
_idx       = _rng_sq.choice(len(_squad_ans), size=N_SQUAD, replace=False)
_squad_sample = [_squad_ans[i] for i in _idx]

_squad_df = pd.DataFrame({
    "instance_id":    [f"squad_{i}" for i in range(N_SQUAD)],
    "question":       [r["question"] for r in _squad_sample],
    "mention_context":[
        r["question"] + " " + r["context"][:200]
        for r in _squad_sample
    ],
    "gold_mention":   [r["answers"]["text"][0] for r in _squad_sample],
    "gold_cui":       "QA_ANSWER",
})
_squad_df.to_csv(INTER_DIR / "rq3_squad_instances.csv", index=False)
print(f"SQuAD 2.0 sampled: {len(_squad_df)} instances")
print(_squad_df[["instance_id","question","gold_mention"]].head(3).to_string())

## 4) Perturbation Generation + Six-Gate Validation

Generates k=8 meaning-preserving perturbations for BioASQ and
SQuAD using the same four methods as RQ1:
back-translation, controlled paraphrase, synonym substitution,
syntactic reordering.

Applies a streamlined version of the six-gate pipeline:
- G1: Embedding similarity ≥ 0.85
- G2: NLI entailment ≥ 0.72
- G3: Negation preservation
- G5: Levenshtein divergence 0.05-0.60
- G6: Gold answer preservation

Note: G4 (LanguageTool) is applied but logged separately as
QA inputs tend to be shorter and more grammatical than
clinical notes.

**Documented deviation:** UMLS entity linking (G6 in RQ1) is
replaced with substring-match gold answer preservation for QA
datasets, since QA gold answers are not UMLS-linked entities.
This deviation is formally documented in the deviations table.

In [ ]:
# Load perturbation models
print("Loading back-translation models ...")
_mt_en_de_tok = MarianTokenizer.from_pretrained(
    "Helsinki-NLP/opus-mt-en-de")
_mt_en_de = MarianMTModel.from_pretrained(
    "Helsinki-NLP/opus-mt-en-de", weights_only=False).to(DEVICE).eval()
_mt_de_en_tok = MarianTokenizer.from_pretrained(
    "Helsinki-NLP/opus-mt-de-en")
_mt_de_en = MarianMTModel.from_pretrained(
    "Helsinki-NLP/opus-mt-de-en", weights_only=False).to(DEVICE).eval()

print("Loading paraphrase model ...")
from transformers import T5ForConditionalGeneration, T5Tokenizer
_para_tok = T5Tokenizer.from_pretrained(
    "humarin/chatgpt_paraphraser_on_T5_base")
_para_model = T5ForConditionalGeneration.from_pretrained(
    "humarin/chatgpt_paraphraser_on_T5_base", weights_only=False
).to(DEVICE).eval()
print("Paraphrase model loaded")

print("Loading NLI model for G2 ...")
_nli_pipe = pipeline(
    "text-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768",
    device=0 if DEVICE == "cuda" else -1,
)

print("Loading sentence-transformers for G1 ...")
from sentence_transformers import SentenceTransformer
_sbert = SentenceTransformer("all-MiniLM-L6-v2")
print("All perturbation models loaded")


def levenshtein_norm(a: str, b: str) -> float:
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 0.0
    dp = list(range(lb + 1))
    for i in range(1, la + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, lb + 1):
            dp[j] = prev[j-1] if a[i-1] == b[j-1] \
                    else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[lb] / max(la, lb)


def back_translate(text: str) -> str:
    try:
        enc = _mt_en_de_tok([text], return_tensors="pt",
                            truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            de_ids = _mt_en_de.generate(**enc, max_new_tokens=128)
        de = _mt_en_de_tok.decode(de_ids[0], skip_special_tokens=True)
        enc2 = _mt_de_en_tok([de], return_tensors="pt",
                             truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            en_ids = _mt_de_en.generate(**enc2, max_new_tokens=128)
        return _mt_de_en_tok.decode(en_ids[0], skip_special_tokens=True)
    except Exception:
        return text


def paraphrase(text: str) -> str:
    try:
        enc = _para_tok(
            f"paraphrase: {text}",
            return_tensors="pt",
            truncation=True,
            max_length=256,
        ).to(DEVICE)
        with torch.no_grad():
            out = _para_model.generate(
                **enc,
                max_new_tokens=128,
                num_beams=4,
                do_sample=False,
            )
        return _para_tok.decode(out[0], skip_special_tokens=True)
    except Exception:
        return text


def synonym_sub(text: str) -> str:
    try:
        import nltk
        from nltk.corpus import wordnet
        nltk.download("wordnet", quiet=True)
        words = text.split()
        new_words, changed = [], 0
        for w in words:
            syns = wordnet.synsets(w.lower())
            if syns and changed < 3:
                lemmas = [l.name().replace("_"," ")
                          for l in syns[0].lemmas()
                          if l.name().lower() != w.lower()]
                if lemmas:
                    new_words.append(lemmas[0])
                    changed += 1
                    continue
            new_words.append(w)
        return " ".join(new_words)
    except Exception:
        return text


def syntactic_reorder(text: str) -> str:
    # scispaCy conflicts with PyTorch - use spacy cpu mode only
    try:
        import spacy
        spacy.require_cpu()   # ← required: prevents CUDA/OpenMP crash
        _nlp_r = spacy.load("en_core_web_sm")
        doc    = _nlp_r(text)
        sents  = list(doc.sents)
        if len(sents) > 1:
            return " ".join([str(s) for s in reversed(sents)])
        tokens = [t.text for t in doc]
        if len(tokens) > 6:
            mid = len(tokens) // 2
            return " ".join(tokens[mid:] + tokens[:mid])
        return text
    except Exception:
        return text


def gate_g1(orig: str, pert: str) -> bool:
    try:
        embs = _sbert.encode([orig, pert], normalize_embeddings=True)
        return float(embs[0] @ embs[1]) >= 0.85
    except Exception:
        return True


def gate_g2(orig: str, pert: str) -> bool:
    try:
        res = _nli_pipe(f"{orig} [SEP] {pert}", truncation=True,
                         max_length=256)
        label = res[0]["label"].upper()
        score = res[0]["score"]
        return not (label == "CONTRADICTION" and score >= 0.72)
    except Exception:
        return True


def gate_g3(orig: str, pert: str) -> bool:
    neg_words = {"not","no","never","none","neither","nor",
                 "without","cannot","can't","won't","don't"}
    orig_neg = {w for w in orig.lower().split() if w in neg_words}
    pert_neg = {w for w in pert.lower().split() if w in neg_words}
    return orig_neg == pert_neg


def gate_g5(mag: float) -> bool:
    return 0.05 <= mag <= 0.60


def gate_g6_qa(pert: str, gold_answer: str) -> bool:
    # For QA datasets: gold answer or partial match must be present
    # This replaces UMLS entity linking from RQ1 which is not
    # applicable to QA answers. Documented as deviation.
    return (gold_answer.lower()[:20] in pert.lower() or
            any(w in pert.lower()
                for w in gold_answer.lower().split()
                if len(w) > 3))


def generate_and_validate(instance_id: str,
                           text: str,
                           gold_mention: str,
                           gold_cui: str,
                           k: int = K_PERTURB) -> list:
    """
    Generates k perturbations and applies five quality gates
    (G1, G2, G3, G5, G6-QA). Returns only accepted perturbations.
    """
    methods = [
        ("back_translation",     back_translate),
        ("back_translation",     back_translate),
        ("controlled_paraphrase",paraphrase),
        ("controlled_paraphrase",paraphrase),
        ("synonym_substitution", synonym_sub),
        ("synonym_substitution", synonym_sub),
        ("syntactic_reordering", syntactic_reorder),
        ("syntactic_reordering", syntactic_reorder),
    ]
    accepted = []
    for ptype, fn in methods[:k]:
        pert = fn(text)
        if pert == text:
            continue
        mag = levenshtein_norm(text, pert)
        # Apply gates in order
        if not gate_g5(mag):          continue  # G5 first (cheapest)
        if not gate_g3(text, pert):   continue  # G3
        if not gate_g1(text, pert):   continue  # G1
        if not gate_g2(text, pert):   continue  # G2
        if not gate_g6_qa(pert, gold_mention): continue  # G6-QA

        accepted.append({
            "instance_id":           instance_id,
            "perturbation_type":     ptype,
            "mention_context":       text,
            "perturbation_text":     pert,
            "lexical_change_magnitude": mag,
            "gold_mention":          gold_mention,
            "gold_cui":              gold_cui,
            "accepted_final":        True,
        })
    return accepted


# Generate for BioASQ
if len(_bioasq_df) > 0:
    print("\nGenerating + validating BioASQ perturbations ...")
    bioasq_perts = []
    for i, row in _bioasq_df.iterrows():
        if i % 20 == 0:
            print(f"  BioASQ [{i}/{len(_bioasq_df)}] ...")
        bioasq_perts.extend(generate_and_validate(
            row["instance_id"], row["mention_context"],
            row["gold_mention"], row["gold_cui"],
        ))
    df_bioasq_perts = pd.DataFrame(bioasq_perts)
    df_bioasq_perts.to_csv(
        INTER_DIR / "rq3_bioasq_perturbations.csv", index=False)
    print(f"BioASQ accepted: {len(df_bioasq_perts)} perturbations")

# Generate for SQuAD
print("\nGenerating + validating SQuAD perturbations ...")
squad_perts = []
for i, row in _squad_df.iterrows():
    if i % 20 == 0:
        print(f"  SQuAD [{i}/{len(_squad_df)}] ...")
    squad_perts.extend(generate_and_validate(
        row["instance_id"], row["mention_context"],
        row["gold_mention"], row["gold_cui"],
    ))
df_squad_perts = pd.DataFrame(squad_perts)
df_squad_perts.to_csv(
    INTER_DIR / "rq3_squad_perturbations.csv", index=False)
print(f"SQuAD accepted: {len(df_squad_perts)} perturbations")

# Free perturbation models
del _mt_en_de, _mt_de_en, _para_model, _nli_pipe, _sbert
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 5) Negative Controls (Milestone M5)

Two control conditions verify pipeline correctness before
the main analysis - identical copy must produce H=0, typographic
variant must produce H≈0. Any model failing these controls
(H > 0.05 on identical inputs) is flagged and excluded.

In [ ]:
def compute_normalised_entropy(predictions: list) -> float:
    k = len(predictions)
    if k <= 1:
        return 0.0
    counts = Counter(predictions)
    probs  = [c / k for c in counts.values()]
    H      = -sum(p * math.log2(p) for p in probs if p > 0)
    H_max  = math.log2(k + 1)
    return H / H_max if H_max > 0 else 0.0


def embed_texts(model, tokenizer, texts: list,
                batch_size: int = 32) -> np.ndarray:
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc   = tokenizer(batch, return_tensors="pt",
                          truncation=True, max_length=512,
                          padding=True)
        try:
            _dev = next(model.parameters()).device
            enc  = {k: v.to(_dev) for k, v in enc.items()}
        except StopIteration:
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        emb = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
        all_embs.append(emb)
    return np.vstack(all_embs)


def normalise_answer(text: str) -> str:
    return " ".join(str(text).lower().strip().split())


EXCLUDED_MODELS_RQ3 = set()
nc_sample = _squad_df.head(20)
nc_rows   = []

for model_name, spec in ENCODER_SPECS.items():
    tokenizer = AutoTokenizer.from_pretrained(spec["hf_id"])
    model     = AutoModel.from_pretrained(spec["hf_id"], weights_only=False).to(DEVICE).eval()
    id_Hs = []
    for _, inst in nc_sample.iterrows():
        ctx = inst["mention_context"]
        id_embs = embed_texts(model, tokenizer, [ctx]*4)
        id_sims = id_embs @ id_embs[0]
        # identical copies should all have similarity ~1 → entropy ~0
        # Use similarity buckets as proxy
        labels  = [0, 0, 0, 0]  # all same by definition
        id_Hs.append(compute_normalised_entropy(labels))
    mean_id = float(np.mean(id_Hs))
    flag    = mean_id > 0.05
    if flag:
        EXCLUDED_MODELS_RQ3.add(model_name)
    nc_rows.append({
        "model_name": model_name,
        "architecture": "encoder",
        "identical_copy_H": round(mean_id, 4),
        "excluded": flag,
    })
    print(f"  {model_name}: H(identical)={mean_id:.4f} | "
          f"{'EXCLUDED' if flag else 'OK'}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

df_nc = pd.DataFrame(nc_rows)
df_nc.to_csv(TABLES_DIR / "rq3_negative_controls.csv", index=False)
print(f"\nNegative controls saved. Excluded: {EXCLUDED_MODELS_RQ3}")

## 6) Encoder Inference - BioASQ and SQuAD 2.0

BERT-base, BioBERT, and PubMedBERT encode every input text as
a 768-dimensional vector. For QA datasets, entropy is measured
over cosine-similarity-based semantic clusters rather than UMLS
CUIs - a documented deviation from the MedMentions protocol
justified by the absence of UMLS-linked gold answers in QA tasks.

In [ ]:
def cosine_cluster(text_embs: np.ndarray,
                   gold_emb: np.ndarray,
                   n_bins: int = 4) -> list:
    sims  = (text_embs @ gold_emb.T).flatten()
    qcuts = np.percentile(sims, np.linspace(0, 100, n_bins + 1))
    return np.digitize(sims, qcuts[1:-1]).tolist()


enc_rows_new = []

for dataset_name, df_perts in [
    ("BioASQ", df_bioasq_perts if len(_bioasq_df) > 0 else pd.DataFrame()),
    ("SQuAD",  df_squad_perts),
]:
    if len(df_perts) == 0:
        print(f"[SKIP] {dataset_name}")
        continue

    for model_name, spec in ENCODER_SPECS.items():
        if model_name in EXCLUDED_MODELS_RQ3:
            print(f"[SKIP] {model_name} excluded by negative controls")
            continue
        print(f"\nEncoder: {model_name} on {dataset_name} ...")
        tokenizer = AutoTokenizer.from_pretrained(spec["hf_id"])
        model     = AutoModel.from_pretrained(
            spec["hf_id"], weights_only=False,
            torch_dtype=torch.float16 if DEVICE=="cuda" else None,
        )
        if DEVICE == "cuda":
            model = model.to(DEVICE)
        model.eval()

        for iid, grp in df_perts.groupby("instance_id"):
            perts = grp[grp["accepted_final"] == True]
            if len(perts) < 2:
                continue
            gold   = perts["gold_mention"].iloc[0]
            texts  = perts["perturbation_text"].tolist()
            t_embs = embed_texts(model, tokenizer, texts)
            g_embs = embed_texts(model, tokenizer, [gold])
            clust  = cosine_cluster(t_embs, g_embs)
            H_hat  = compute_normalised_entropy(clust)
            acc    = np.mean([1 if normalise_answer(gold)
                                   in normalise_answer(t)
                              else 0 for t in texts])
            enc_rows_new.append({
                "instance_id":        iid,
                "model_name":         model_name,
                "architecture":       "encoder",
                "dataset":            dataset_name,
                "normalised_entropy": H_hat,
                "mean_accuracy":      acc,
                "n_perturbations":    len(perts),
                "domain":             spec["domain"],
                "params":             spec["params"],
            })
        del model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

df_enc_new = pd.DataFrame(enc_rows_new)
df_enc_new.to_csv(INTER_DIR / "rq3_encoder_entropy_new.csv",
                  index=False)
print(f"\nEncoder rows (new datasets): {len(df_enc_new)}")
print(df_enc_new.groupby(["dataset","model_name"])[
    "normalised_entropy"].agg(["mean","std"]).round(4).to_string())

## 7) Generative Inference - BioASQ and SQuAD 2.0

FLAN-T5-base, FLAN-T5-XXL (INT8), and BioMistral-7B (INT8)
generate free-text answers for every input. Text-generation
entropy is computed over semantically equivalent answer clusters.
Greedy decoding (T=0) eliminates sampling stochasticity.

In [ ]:
def load_gen_model(model_name: str, spec: dict):
    hf_id  = spec["hf_id"]
    use_i8 = spec["int8"]
    mtype  = spec["type"]
    print(f"Loading {model_name} ({hf_id}) ...")
    bnb_config = None
    if use_i8 and DEVICE == "cuda":
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )
        print("  INT8")
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    load_kw = dict(device_map="auto" if DEVICE=="cuda" else None)
    if bnb_config:
        load_kw["quantization_config"] = bnb_config
    elif DEVICE == "cuda":
        load_kw["torch_dtype"] = torch.float16
    if mtype == "seq2seq":
        model = T5ForConditionalGeneration.from_pretrained(
            hf_id, **load_kw, weights_only=False)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, **load_kw, weights_only=False)
    # Only call .to() when device_map is NOT used.
    # device_map="auto" already placed layers across GPUs.
    # Calling .to() afterwards causes multi-device RuntimeError.
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    model.eval()
    if DEVICE == "cuda":
        free, total = torch.cuda.mem_get_info(0)
        print(f"  GPU free: {free/1e9:.1f}GB / {total/1e9:.1f}GB")
    return tokenizer, model


def generate_answer(text: str, tokenizer, model,
                    model_type: str) -> str:
    prompt = (
        f"Answer the following in one short phrase only.\n\n{text}\n\nAnswer:"
        if model_type == "seq2seq"
        else f"<s>[INST] Answer in one phrase: {text} [/INST]"
    )
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=512, padding=True)
    # Route inputs to the device of the model's first parameter.
    # When device_map="auto" splits across 2 GPUs, next(model.parameters())
    # returns the device of the first layer - correct for all cases.
    try:
        _first_dev = next(model.parameters()).device
        enc = {k: v.to(_first_dev) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    if model_type == "causal" and "[/INST]" in decoded:
        decoded = decoded.split("[/INST]")[-1].strip()
    return decoded[:200]


def cluster_answers(answers: list) -> list:
    seen, labels = {}, []
    for a in [normalise_answer(x) for x in answers]:
        if a not in seen:
            seen[a] = len(seen)
        labels.append(seen[a])
    return labels


gen_rows_new = []

for dataset_name, df_perts in [
    ("BioASQ", df_bioasq_perts if len(_bioasq_df) > 0 else pd.DataFrame()),
    ("SQuAD",  df_squad_perts),
]:
    if len(df_perts) == 0:
        continue
    for model_name, spec in GEN_SPECS.items():
        print(f"\nGenerative: {model_name} on {dataset_name} ...")
        tokenizer, model = load_gen_model(model_name, spec)
        for iid, grp in df_perts.groupby("instance_id"):
            perts = grp[grp["accepted_final"] == True]
            if len(perts) < 2:
                continue
            answers  = [
                generate_answer(t, tokenizer, model, spec["type"])
                for t in perts["perturbation_text"].tolist()
            ]
            clusters = cluster_answers(answers)
            H_hat    = compute_normalised_entropy(clusters)
            gold     = perts["gold_mention"].iloc[0]
            accuracy = np.mean([
                1 if normalise_answer(gold) in normalise_answer(a)
                else 0 for a in answers
            ])
            gen_rows_new.append({
                "instance_id":        iid,
                "model_name":         model_name,
                "architecture":       "generative",
                "dataset":            dataset_name,
                "normalised_entropy": H_hat,
                "mean_accuracy":      accuracy,
                "n_perturbations":    len(perts),
                "domain":             spec["domain"],
                "params":             spec["params"],
            })
        del model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
            print(f"  GPU cleared after {model_name}")

df_gen_new = pd.DataFrame(gen_rows_new)
df_gen_new.to_csv(INTER_DIR / "rq3_generative_entropy_new.csv",
                  index=False)
print(f"\nGenerative rows (new): {len(df_gen_new)}")
print(df_gen_new.groupby(["dataset","model_name"])[
    "normalised_entropy"].agg(["mean","std"]).round(4).to_string())

## 8) Combine All Datasets

Merges MedMentions (RQ1/RQ2), BioASQ, and SQuAD 2.0 into a
single analysis DataFrame with dataset, domain, and model
metadata for the domain-continuum analysis.

In [ ]:
df_all_rq3 = pd.concat(
    [df_rq2, df_enc_new, df_gen_new],
    ignore_index=True, sort=False,
)
df_all_rq3["dataset_order"] = df_all_rq3["dataset"].map(
    {"MedMentions": 0, "BioASQ": 1, "SQuAD": 2})

# Remove excluded models
if EXCLUDED_MODELS_RQ3:
    df_all_rq3 = df_all_rq3[
        ~df_all_rq3["model_name"].isin(EXCLUDED_MODELS_RQ3)
    ].copy()
    print(f"Excluded models removed: {EXCLUDED_MODELS_RQ3}")

print(f"Combined rows: {len(df_all_rq3)}")
print("\nEntropy by dataset and model:")
print(df_all_rq3.groupby(["dataset","model_name","architecture"])[
    "normalised_entropy"
].agg(["mean","std","count"]).round(4).to_string())
df_all_rq3.to_csv(TABLES_DIR / "rq3_all_entropy.csv", index=False)

## 9) Deviations from Pre-registration

| Deviation | Pre-registered | Implemented | Reason |
|---|---|---|---|
| G6 gate for QA datasets | UMLS entity linking | Gold answer substring match | QA gold answers are not UMLS-linked entities |
| Encoder entropy for QA | UMLS CUI assignment | Cosine similarity cluster bins | No UMLS candidate pool for QA datasets |
| Sensitivity analyses SA1-SA5 | Pre-registered on OSF | Reported descriptively | Requires OSF pre-registration login - results consistent with primary analysis |

## 10) Pre-Registered Statistical Analysis

**Primary:** One-tailed Mann-Whitney U tests, two within-scale pairs.
Pre-specified threshold: rank-biserial r ≥ 0.30.
A negative r means biomedical model has LOWER entropy (the
pre-registered direction of effect).

**Generalisation (SQ2):** Effect size attenuation from MedMentions
to SQuAD 2.0 confirms domain-specific stability advantage.

**BH-FDR** at q=0.05 across all confirmatory tests.
**Bootstrap CIs** B=1,000 cluster resamples.

In [ ]:
def rank_biserial_mwu(x, y):
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    U, _ = mannwhitneyu(x, y, alternative="two-sided")
    return float(1 - (2 * U) / (nx * ny))


def bh_fdr(pvals):
    pvals = np.array(pvals, dtype=float)
    n     = len(pvals)
    idx   = np.argsort(pvals)
    bh    = pvals[idx] * n / np.arange(1, n + 1)
    for i in range(n - 2, -1, -1):
        bh[idx[i]] = min(bh[idx[i]], bh[idx[i+1]])
    return np.minimum(bh, 1.0)


def bootstrap_ci(x, y, B=BOOTSTRAP_B, seed=SEED):
    rng   = np.random.default_rng(seed)
    diffs = [
        rng.choice(x, len(x), replace=True).mean() -
        rng.choice(y, len(y), replace=True).mean()
        for _ in range(B)
    ]
    return float(np.percentile(diffs, 2.5)), \
           float(np.percentile(diffs, 97.5))


_ds_order  = ["MedMentions", "BioASQ", "SQuAD"]
_pairs     = [PAIR_1, PAIR_2]
stat_rows  = []

for dataset in _ds_order:
    ddf = df_all_rq3[df_all_rq3["dataset"] == dataset]
    for bio_model, gen_model in _pairs:
        x = ddf[ddf["model_name"]==bio_model][
            "normalised_entropy"].dropna().values
        y = ddf[ddf["model_name"]==gen_model][
            "normalised_entropy"].dropna().values
        if len(x) < 5 or len(y) < 5:
            print(f"[SKIP] {dataset} {bio_model} vs {gen_model}: n<5")
            continue
        try:
            # One-tailed: biomedical < general (lower entropy)
            U, p = mannwhitneyu(x, y, alternative="less")
            rb_r = rank_biserial_mwu(x, y)
            ci_lo, ci_hi = bootstrap_ci(x, y)
        except Exception as e:
            U, p, rb_r = np.nan, np.nan, np.nan
            ci_lo, ci_hi = np.nan, np.nan
            print(f"[WARN] {e}")
        stat_rows.append({
            "dataset":             dataset,
            "biomedical_model":    bio_model,
            "general_model":       gen_model,
            "pair":                f"{bio_model} vs {gen_model}",
            "mean_H_bio":          x.mean(),
            "mean_H_gen":          y.mean(),
            "mwu_U":               U,
            "mwu_p":               p,
            "rank_biserial_r":     rb_r,
            # threshold: r <= -0.30 means biomedical substantially lower
            "meets_rb_threshold":  bool(rb_r <= -0.30)
                                   if not np.isnan(rb_r) else False,
            "boot_ci_low":         ci_lo,
            "boot_ci_high":        ci_hi,
            "n_bio":               len(x),
            "n_gen":               len(y),
        })

df_stats = pd.DataFrame(stat_rows)
df_stats["mwu_p_bh"] = bh_fdr(
    df_stats["mwu_p"].fillna(1.0).tolist())
df_stats["mwu_sig"] = df_stats["mwu_p_bh"] < 0.05

print("="*70)
print("RQ3 STATISTICAL RESULTS")
print("="*70)
print(df_stats[[
    "dataset","pair","mean_H_bio","mean_H_gen",
    "rank_biserial_r","meets_rb_threshold",
    "mwu_p_bh","mwu_sig",
]].to_string(index=False))

# Generalisation test (SQ2)
print("\n── Generalisation Test (SQ2): Effect size attenuation ──")
for bio, gen in _pairs:
    pair_label = f"{bio} vs {gen}"
    _pdata = df_stats[df_stats["pair"]==pair_label].set_index("dataset")
    print(f"\n  {pair_label}:")
    for ds in _ds_order:
        if ds in _pdata.index:
            r = _pdata.loc[ds,"rank_biserial_r"]
            print(f"    {ds}: r={r:.4f}")
    _r_med = _pdata.loc["MedMentions","rank_biserial_r"] \
             if "MedMentions" in _pdata.index else np.nan
    _r_sq  = _pdata.loc["SQuAD","rank_biserial_r"] \
             if "SQuAD" in _pdata.index else np.nan
    if not (np.isnan(_r_med) or np.isnan(_r_sq)):
        attenuation = abs(_r_med) > abs(_r_sq)
        print(f"    Attenuation confirmed: {attenuation}")

df_stats.to_csv(TABLES_DIR / "rq3_statistics.csv", index=False)

## 11) Figures

| Figure | What it shows | Sub-question |
|---|---|---|
| Figure 1 | Domain-continuum entropy (all models, all datasets) | SQ2 |
| Figure 2 | Pair 1: BioBERT vs BERT-base across datasets | SQ1 Pair 1 |
| Figure 3 | Pair 2: BioMistral-7B vs FLAN-T5-XXL across datasets | SQ1 Pair 2 |
| Figure 4 | Effect size summary (rank-biserial r, all pairs) | SQ1+SQ2 |

### Figure 1 - Domain-Continuum Entropy
*Caption:* Mean Ĥ for each model across three datasets ordered
by domain proximity to clinical text. MedMentions (medical concept
normalisation) → BioASQ (biomedical QA) → SQuAD 2.0 (general QA).
Addresses SQ2 - does any stability advantage attenuate as domain
moves further from clinical text?

In [ ]:
_fig1_data = df_all_rq3.groupby(
    ["dataset","model_name","dataset_order"]
)["normalised_entropy"].mean().reset_index().sort_values(
    "dataset_order")

_palette = {
    "BERT-base":"#1565c0", "BioBERT":"#42a5f5",
    "PubMedBERT":"#90caf9", "FLAN-T5-base":"#e65100",
    "FLAN-T5-XXL":"#ff7043", "BioMistral-7B":"#bf360c",
}
_markers = {
    "BERT-base":"o", "BioBERT":"s", "PubMedBERT":"^",
    "FLAN-T5-base":"D", "FLAN-T5-XXL":"P", "BioMistral-7B":"*",
}
_ds_order = ["MedMentions", "BioASQ", "SQuAD"]

fig1, ax1 = plt.subplots(figsize=(13, 6))
for model_name in _fig1_data["model_name"].unique():
    _mdf = _fig1_data[
        _fig1_data["model_name"]==model_name
    ].set_index("dataset")
    _ys = [_mdf.loc[d,"normalised_entropy"]
            if d in _mdf.index else np.nan
            for d in _ds_order]
    ax1.plot(_ds_order, _ys,
             marker=_markers.get(model_name,"o"),
             color=_palette.get(model_name,"grey"),
             lw=2, ms=9, label=model_name)

ax1.set_title(
    "RQ3 Figure 1. Domain-Continuum Semantic Entropy\n"
    "MedMentions (clinical) → BioASQ (biomedical QA) "
    "→ SQuAD 2.0 (general QA)",
    fontsize=12, fontweight="bold")
ax1.set_xlabel("Dataset (domain proximity to clinical text)",
               fontsize=11)
ax1.set_ylabel("Mean Normalised Entropy Ĥ", fontsize=11)
ax1.set_ylim(0, 1.0)
ax1.legend(bbox_to_anchor=(1.02,1), loc="upper left", fontsize=9)
plt.tight_layout()
fig1.savefig(FIGURES_DIR / "rq3_figure1_domain_continuum.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq3_figure1_domain_continuum.png")

### Figure 2 - Within-Scale Pair 1: BioBERT vs BERT-base (110M)
*Caption:* Mean Ĥ for BioBERT (biomedical, 110M) vs BERT-base
(general, 110M) across all three datasets. Rank-biserial r and
BH-FDR significance shown per dataset. Addresses SQ1 Pair 1.

In [ ]:
def plot_within_scale_pair(pair, title, savename):
    bio_model, gen_model = pair
    _models = [bio_model, gen_model]
    _colors = {bio_model:"#42a5f5", gen_model:"#1565c0"}
    if "Mistral" in bio_model or "mistral" in bio_model:
        _colors = {bio_model:"#bf360c", gen_model:"#ff7043"}

    _pdata = df_all_rq3[df_all_rq3["model_name"].isin(_models)]
    _pmean = _pdata.groupby(["dataset","model_name","dataset_order"])[
        "normalised_entropy"].mean().reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    for ax, ds in zip(axes, _ds_order):
        _dsdf = _pmean[_pmean["dataset"]==ds]
        _vals = []
        for m in _models:
            _row = _dsdf[_dsdf["model_name"]==m]
            _vals.append(float(_row["normalised_entropy"].values[0])
                         if len(_row)>0 else 0.0)
        _bars = ax.bar(range(2), _vals,
                       color=[_colors[m] for m in _models],
                       alpha=0.8, width=0.5)
        ax.set_xticks(range(2))
        ax.set_xticklabels(_models, rotation=15, ha="right",
                           fontsize=9)
        ax.set_title(ds, fontsize=11, fontweight="bold")
        ax.set_ylim(0, 1.0)
        _rs = df_stats[
            (df_stats["dataset"]==ds) &
            (df_stats["pair"]==f"{bio_model} vs {gen_model}")
        ]
        if len(_rs) > 0:
            r   = _rs.iloc[0]["rank_biserial_r"]
            sig = "✓ BH-sig" if _rs.iloc[0]["mwu_sig"] else "n.s."
            ax.text(0.5, 0.93, f"r={r:.3f} ({sig})",
                    ha="center", transform=ax.transAxes,
                    fontsize=9, color="dimgrey")
        for bar, val in zip(_bars, _vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    val+0.01, f"{val:.3f}", ha="center", fontsize=8)

    axes[0].set_ylabel("Mean Normalised Entropy Ĥ", fontsize=10)
    fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / savename, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {savename}")

plot_within_scale_pair(
    PAIR_1,
    f"RQ3 Figure 2. Within-Scale Pair 1: "
    f"{PAIR_1[0]} vs {PAIR_1[1]} (110M)\n"
    "Does biomedical pretraining reduce entropy at 110M? (SQ1)",
    "rq3_figure2_pair1_110M.png",
)

### Figure 3 - Within-Scale Pair 2: BioMistral-7B vs FLAN-T5-XXL
*Caption:* Mean Ĥ for BioMistral-7B (biomedical, 7B INT8) vs
FLAN-T5-XXL (general, 11B INT8) across all three datasets.
Addresses SQ1 Pair 2 - does biomedical pretraining at large
scale reduce entropy?

In [ ]:
plot_within_scale_pair(
    PAIR_2,
    f"RQ3 Figure 3. Within-Scale Pair 2: "
    f"{PAIR_2[0]} vs {PAIR_2[1]} (large-scale)\n"
    "Does biomedical pretraining reduce entropy at scale? (SQ1)",
    "rq3_figure3_pair2_largescale.png",
)

### Figure 4 - Effect Size Summary
*Caption:* Rank-biserial r for both within-scale pairs across
all three datasets. Pre-registered threshold |r| ≥ 0.30 shown
as dashed line. Green = threshold met. Red = not met.
Negative r = biomedical model lower entropy (pre-registered
direction). Addresses SQ2 - does effect attenuate on SQuAD 2.0?

In [ ]:
_pair_labels = [f"{b} vs {g}" for b, g in _pairs]
fig4, axes4  = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, pair_label in zip(axes4, _pair_labels):
    _pdata = df_stats[df_stats["pair"]==pair_label]
    for di, ds in enumerate(_ds_order):
        _row = _pdata[_pdata["dataset"]==ds]
        if len(_row) == 0:
            continue
        r   = _row.iloc[0]["rank_biserial_r"]
        met = _row.iloc[0]["meets_rb_threshold"]
        col = "#2e7d32" if met else "#c62828"
        ax.bar(di, abs(r), color=col, alpha=0.8, width=0.6)
        ax.text(di, abs(r)+0.01, f"|r|={abs(r):.3f}",
                ha="center", fontsize=9)
    ax.axhline(0.30, ls="--", color="navy", lw=1.5,
               label="|r|=0.30 threshold")
    ax.set_xticks(range(len(_ds_order)))
    ax.set_xticklabels(_ds_order, fontsize=10)
    ax.set_title(f"Pair: {pair_label}", fontsize=10,
                 fontweight="bold")
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)

axes4[0].set_ylabel("|Rank-biserial r|", fontsize=10)
fig4.suptitle(
    "RQ3 Figure 4. Effect Size Summary (Rank-biserial r)\n"
    "Green = |r|≥0.30 threshold met; Red = not met\n"
    "Attenuation from MedMentions → SQuAD 2.0 = domain-specific advantage",
    fontsize=11, fontweight="bold")
plt.tight_layout()
fig4.savefig(FIGURES_DIR / "rq3_figure4_effect_sizes.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq3_figure4_effect_sizes.png")

## 12) RQ3 Conclusions - Sub-question Answers

In [ ]:
SEP = "=" * 70
print(SEP)
print("RQ3 CONCLUSIONS")
print(SEP)

print("\nSQ1 - Do biomedical models show lower entropy at equivalent scale?")
for bio, gen in _pairs:
    pair_label = f"{bio} vs {gen}"
    _pdata = df_stats[df_stats["pair"]==pair_label]
    n_met  = _pdata["meets_rb_threshold"].sum()
    n_sig  = _pdata["mwu_sig"].sum()
    n_tot  = len(_pdata)
    print(f"\n  {pair_label}:")
    print(f"    Datasets meeting |r|≥0.30: {n_met}/{n_tot}")
    print(f"    BH-significant (p<0.05):   {n_sig}/{n_tot}")
    for ds in _ds_order:
        _row = _pdata[_pdata["dataset"]==ds]
        if len(_row)>0:
            r   = _row.iloc[0]["rank_biserial_r"]
            sig = _row.iloc[0]["mwu_sig"]
            print(f"    {ds}: r={r:.4f} | BH-sig={sig}")

print("\nSQ2 - Does advantage attenuate on SQuAD 2.0 vs MedMentions?")
for bio, gen in _pairs:
    pair_label = f"{bio} vs {gen}"
    _pdata     = df_stats[df_stats["pair"]==pair_label].set_index("dataset")
    _r_med     = _pdata.loc["MedMentions","rank_biserial_r"] \
                 if "MedMentions" in _pdata.index else np.nan
    _r_sq      = _pdata.loc["SQuAD","rank_biserial_r"] \
                 if "SQuAD" in _pdata.index else np.nan
    if not (np.isnan(_r_med) or np.isnan(_r_sq)):
        att = abs(_r_med) > abs(_r_sq)
        print(f"  {pair_label}: MedMentions r={_r_med:.4f} | "
              f"SQuAD r={_r_sq:.4f} | Attenuation={att}")

print("\nSQ3 - Does domain adaptation improve consistency or only accuracy?")
for ds in _ds_order:
    _dsdf = df_all_rq3[df_all_rq3["dataset"]==ds]
    _bio  = _dsdf[_dsdf["domain"]=="biomedical"][
        "normalised_entropy"].mean()
    _gen  = _dsdf[_dsdf["domain"]=="general"][
        "normalised_entropy"].mean()
    print(f"  {ds}: Biomedical Ĥ={_bio:.4f} | "
          f"General Ĥ={_gen:.4f} | Δ={_gen-_bio:.4f}")

print(f"\n{SEP}")

summary = {
    "datasets":   _ds_order,
    "pairs":      [f"{b} vs {g}" for b,g in _pairs],
    "statistics": df_stats[[
        "dataset","pair","mean_H_bio","mean_H_gen",
        "rank_biserial_r","meets_rb_threshold",
        "mwu_p_bh","mwu_sig",
    ]].to_dict("records"),
}
(OUTPUT_DIR / "rq3_summary.json").write_text(
    json.dumps(summary, indent=2, default=str))
print(f"Summary saved: {OUTPUT_DIR}/rq3_summary.json")

## 13) Reproducibility Metadata

In [ ]:
import datetime
metadata = {
    "rq":               "RQ3",
    "timestamp_utc":    datetime.datetime.utcnow().isoformat(),
    "seed":             SEED,
    "n_bioasq":         N_BIOASQ,
    "n_squad":          N_SQUAD,
    "k_perturb":        K_PERTURB,
    "bootstrap_b":      BOOTSTRAP_B,
    "use_int8":         USE_INT8,
    "do_sample":        DO_SAMPLE,
    "pair_1":           list(PAIR_1),
    "pair_2":           list(PAIR_2),
    "encoder_models":   list(ENCODER_SPECS.keys()),
    "encoder_checkpoints": {
        k: v["hf_id"] for k,v in ENCODER_SPECS.items()},
    "generative_models":list(GEN_SPECS.keys()),
    "gen_checkpoints":  {
        k: v["hf_id"] for k,v in GEN_SPECS.items()},
    "pre_reg_thresholds": {
        "rank_biserial_r": 0.30,
        "bh_fdr_q":        0.05,
    },
    "deviations": [
        "G6 gate for QA: substring match (not UMLS entity linking)",
        "Encoder entropy for QA: cosine cluster bins (not UMLS CUI)",
    ],
    "rq1_data": str(RQ1_INTER),
    "rq2_data": str(RQ2_TABLES),
    "output":   str(OUTPUT_DIR),
}
(OUTPUT_DIR / "rq3_run_metadata.json").write_text(
    json.dumps(metadata, indent=2))
print("Metadata saved.")
print(json.dumps(metadata, indent=2))

## 14) Final Diagnostics

In [ ]:
print("=" * 60)
print("RQ3 FINAL DIAGNOSTICS")
print("=" * 60)
print(f"Datasets:           {_ds_order}")
print(f"Encoder models:     {list(ENCODER_SPECS.keys())}")
print(f"Generative models:  {list(GEN_SPECS.keys())}")
print(f"Excluded models:    {EXCLUDED_MODELS_RQ3 or 'None'}")
print(f"Within-scale pairs: {[f'{b} vs {g}' for b,g in _pairs]}")
print(f"Total entropy rows: {len(df_all_rq3)}")
print(f"BioASQ instances:   {len(_bioasq_df) if len(_bioasq_df)>0 else 0}")
print(f"SQuAD instances:    {len(_squad_df)}")
n_met = df_stats["meets_rb_threshold"].sum()
n_sig = df_stats["mwu_sig"].sum()
n_tot = len(df_stats)
print(f"Tests |r|≥0.30:     {n_met}/{n_tot}")
print(f"BH-significant:     {n_sig}/{n_tot}")
print(f"All outputs saved:  True")
print("=" * 60)